# Data Visualization with Plotly Express

In this notebook, we will learn how to explore a dataset with **Plotly Express** (`px`), the simplest way to create interactive charts in Python.

A good chart always starts with a **question**. For each type of question, we will see:
1. Which chart answers it
2. The Plotly Express code
3. How to interpret the result

| Question | Chart | Function |
|---|---|---|
| How is one variable distributed? | Histogram | `px.histogram` |
| Does a number differ between groups? | Box plot | `px.box` |
| What are the proportions of each category? | Pie / bar chart | `px.pie`, `px.bar` |
| Are two numbers related? | Scatter plot | `px.scatter` |
| How does an average or a rate change between groups? | Bar chart | `px.bar` |
| How does a value evolve over time? | Line chart | `px.line` |

## 1. Getting started

### Import and load the data
Plotly Express comes with some sample datasets. We will use **tips**: each row is a restaurant bill, with the amount, the tip, and information about the customers.

In [ ]:
import pandas as pd
import plotly.express as px

tips = px.data.tips()
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [ ]:
tips.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    str    
 3   smoker      244 non-null    str    
 4   day         244 non-null    str    
 5   time        244 non-null    str    
 6   size        244 non-null    int64  
dtypes: float64(2), int64(1), str(4)
memory usage: 13.5 KB


### How a Plotly Express call works
Every Plotly Express function follows the same pattern:

```python
fig = px.chart_type(dataframe, x="column", y="column", color="column", title="...")
fig.show()
```

- `dataframe`: the pandas DataFrame containing the data
- `x`, `y`: the **names of the columns** to put on each axis
- `color`: a column used to split the data into colored groups
- `title`, `labels`: to make the chart readable

Once you know this pattern, you know how to use almost every function of Plotly Express.

In [ ]:
fig = px.scatter(tips, x="total_bill", y="tip",
                 title="Tip vs total bill",
                 labels={"total_bill": "Total bill ($)", "tip": "Tip ($)"})
fig.show()

**Interpretation:**
Plotly charts are **interactive**: hover over a point to see its values, drag to zoom, double-click to reset. The `labels` argument renames the axes without changing the DataFrame.

## 2. How is one variable distributed? → `px.histogram`

A histogram cuts a numeric variable into intervals (*bins*) and counts how many rows fall into each interval.

In [ ]:
fig = px.histogram(tips, x="total_bill", title="Distribution of the total bill")
fig.show()

**Interpretation:** Most bills are between 10 and 20 dollars. The distribution is *skewed to the right*: there are a few very high bills.

### Choosing the number of bins
Use `nbins` to make the intervals smaller or larger.

In [ ]:
fig = px.histogram(tips, x="total_bill", nbins=10, title="Distribution of the total bill (10 bins)")
fig.show()

### Comparing the distribution between groups
Add `color=` to split the histogram by category. With `barmode="overlay"`, the histograms are drawn on top of each other, which makes the comparison easier.

In [ ]:
fig = px.histogram(tips, x="total_bill", color="sex", barmode="overlay",
                   title="Distribution of the total bill by sex")
fig.show()

**Interpretation:** We can compare the shape of the two distributions. Be careful: there are more men than women in the dataset, so the male bars are higher. We compare **shapes**, not heights.

A histogram can also count a **categorical** column: each bar is then the number of rows of each category.

## 3. Does a number differ between groups? → `px.box`

A box plot summarizes a distribution in a few values:
- the line in the middle of the box is the **median**
- the box goes from the first quartile (25%) to the third quartile (75%)
- the points outside the whiskers are **outliers**

It is the best chart to compare a numeric variable across several groups.

In [ ]:
fig = px.box(tips, x="day", y="total_bill", title="Total bill by day")
fig.show()

**Interpretation:** The median bill is higher on Saturday and Sunday than during the week. Hover over a box to read the exact median and quartiles.

### Adding a second group
Use `color=` to split each box into sub-groups.

In [ ]:
fig = px.box(tips, x="day", y="total_bill", color="sex",
             title="Total bill by day and by sex")
fig.show()

**Interpretation:** For every day, we can now compare men and women. A histogram with `color=` and a box plot with `color=` are **two different ways** of answering the same question: "is the distribution different between groups?"

## 4. What are the proportions of each category? → `px.pie` / `px.bar`

To show proportions, we first need to **count** each category. This is done with `value_counts()`.

In [ ]:
day_counts = tips["day"].value_counts().reset_index()
day_counts

,day,count
0,Sat,87
1,Sun,76
2,Thur,62
3,Fri,19


`reset_index()` turns the result into a DataFrame with two columns:
- `day`: the categories
- `count`: the number of rows of each category

We can now give these two columns to `px.pie`: `names` for the categories, `values` for the counts.

In [ ]:
fig = px.pie(day_counts, names="day", values="count", title="Share of bills per day")
fig.show()

**Interpretation:** The pie chart directly shows percentages. Saturday is the busiest day, with more than a third of the bills.

**Shortcut:** if you give `px.pie` the original DataFrame with only `names`, it counts the rows by itself.

In [ ]:
fig = px.pie(tips, names="smoker", title="Share of smokers")
fig.show()

### Getting the percentages as numbers
Use `value_counts(normalize=True)` to get proportions instead of counts.

In [ ]:
tips["smoker"].value_counts(normalize=True) * 100

smoker
No     61.885246
Yes    38.114754
Name: proportion, dtype: float64

**Pie or bar?** A pie chart is fine for 2 to 5 categories. With more categories, a bar chart is easier to read.

## 5. Are two numbers related? → `px.scatter`

A scatter plot shows each row as a point. If the points follow a direction, the two variables are correlated.

In [ ]:
fig = px.scatter(tips, x="total_bill", y="tip", title="Tip vs total bill")
fig.show()

**Interpretation:** The points go up from left to right: the higher the bill, the higher the tip. The two variables are **positively correlated**.

We can measure this relation with the correlation coefficient (between -1 and 1):

In [ ]:
tips["total_bill"].corr(tips["tip"])

np.float64(0.6757341092113645)

A value of 0.68 means a fairly strong positive correlation.

### Adding more information
- `color=`: color the points by group
- `size=`: change the size of the points according to a column
- `hover_data=`: add columns to the information shown on hover
- `trendline="ols"`: add a regression line

In [ ]:
fig = px.scatter(tips, x="total_bill", y="tip", color="sex", size="size",
                 hover_data=["day", "time"], trendline="ols",
                 title="Tip vs total bill, by sex")
fig.show()

**Interpretation:** The trend lines of men and women are very close: the relation between bill and tip is similar for both groups. Hover over a point to see the day and the time of the meal.

## 6. How does an average or a rate change between groups? → `groupby` + `px.bar`

Sometimes the raw data is not what we want to plot. We first need to **aggregate** it, for example by computing the average per group, then plot the result.

### Average per group

In [ ]:
mean_tip = tips.groupby("day")["tip"].mean().reset_index()
mean_tip

,day,tip
0,Fri,2.734737
1,Sat,2.993103
2,Sun,3.255132
3,Thur,2.771452


In [ ]:
fig = px.bar(mean_tip, x="day", y="tip", title="Average tip per day")
fig.show()

**Interpretation:** The average tip is a bit higher on Sunday.

### Rate of a Yes/No column per group
What is the **share of smokers** each day? `smoker` contains text (`"Yes"` / `"No"`), so we cannot compute a mean directly.

The trick: convert `"Yes"` to 1 and `"No"` to 0. The mean of a 0/1 column is then the **proportion of Yes**.

In [ ]:
tips["smoker_num"] = tips["smoker"].map({"Yes": 1, "No": 0})
tips.head()

,total_bill,tip,sex,smoker,day,time,size,smoker_num
0,16.99,1.01,Female,No,Sun,Dinner,2,0
1,10.34,1.66,Male,No,Sun,Dinner,3,0
2,21.01,3.50,Male,No,Sun,Dinner,3,0
3,23.68,3.31,Male,No,Sun,Dinner,2,0
4,24.59,3.61,Female,No,Sun,Dinner,4,0


In [ ]:
smoker_rate = tips.groupby("day")["smoker_num"].mean().reset_index()
smoker_rate

,day,smoker_num
0,Fri,0.789474
1,Sat,0.482759
2,Sun,0.250000
3,Thur,0.274194


In [ ]:
fig = px.bar(smoker_rate, x="day", y="smoker_num", title="Share of smokers per day",
             labels={"smoker_num": "Share of smokers"})
fig.show()

**Interpretation:** Almost 80% of the Friday customers are smokers, against only about 25% on Thursday and Sunday.

### Grouped bars
With `color=` and `barmode="group"`, you can compare two groupings in the same chart.

In [ ]:
mean_bill = tips.groupby(["day", "sex"])["total_bill"].mean().reset_index()

fig = px.bar(mean_bill, x="day", y="total_bill", color="sex", barmode="group",
             title="Average bill per day and by sex")
fig.show()

## 7. How does a value evolve over time? → `px.line`

For time series, we use a line chart. Let's load another sample dataset, **gapminder**, with the life expectancy, population and GDP of every country from 1952 to 2007.

In [ ]:
gapminder = px.data.gapminder()
gapminder.head()

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


In [ ]:
france = gapminder[gapminder["country"] == "France"]

fig = px.line(france, x="year", y="lifeExp", title="Life expectancy in France")
fig.show()

**Interpretation:** Life expectancy in France went from about 67 years in 1952 to more than 80 years in 2007.

With `color=`, we draw one line per group:

In [ ]:
countries = gapminder[gapminder["country"].isin(["France", "Japan", "Brazil", "Nigeria"])]

fig = px.line(countries, x="year", y="lifeExp", color="country",
              title="Life expectancy over time")
fig.show()

**Interpretation:** All four countries progress, but at different speeds. Japan overtook France in the 1970s, while Nigeria remains far behind.

## Summary
- Start from a **question**, then choose the chart
- All Plotly Express functions share the same pattern: `px.chart(df, x=..., y=..., color=...)`
- `color=` is the easiest way to compare groups
- To plot proportions or averages, **aggregate first** (`value_counts`, `groupby().mean()`), then plot the result